In [ ]:
# Configuración COSMIC v0.0.1 - Estructura Modular
import sys
from pathlib import Path

# Configuración automática de rutas relativas
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parents[2]  # Tres niveles arriba desde data/test/NGC6383/

# Agregar al path si no está
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"🌟 COSMIC v0.0.1 - NGC6383 Analysis")
print(f"📁 Directorio actual: {CURRENT_DIR}")
print(f"📁 Proyecto: {PROJECT_ROOT}")

# Verificar instalación de COSMIC
try:
    import cosmic
    print("✅ COSMIC modular disponible")
except ImportError:
    print("⚠️  Instalando COSMIC...")
    import os
    os.system(f"pip install -e {PROJECT_ROOT}")
    import cosmic
    print("✅ COSMIC instalado")

In [ ]:
# Imports usando la nueva estructura modular
from cosmic.analysis.analyzer import ClusterAnalyzer

# El import legacy sigue funcionando para compatibilidad:
# from COSMIC import ClusterAnalyzer  # Funciona igual

print(f"📦 ClusterAnalyzer importado: {ClusterAnalyzer}")
print("✅ Listo para análisis con nueva API modular")

In [ ]:
# Configuración de datos usando rutas relativas
data_folder = 'data/40/'
file_path = data_folder + 'clustering_results.dill'

print(f"📁 Carpeta de datos: {data_folder}")
print(f"📄 Archivo: {file_path}")
print(f"📊 Existe: {Path(file_path).exists()}")

# Inicializar el analizador con la nueva API
ca = ClusterAnalyzer(file_path)
print("✅ ClusterAnalyzer inicializado correctamente")

In [ ]:
# 3) Build (or retrieve) a summary table: one row per cluster, 
#    with columns for label, n_members, persistence, etc.
ca.clusters_summary(include_noise=True)

In [ ]:
ca.plot_persistence_vs_members(percentile=0.8, figsize=(10,10))

In [ ]:
cluster_id = 12
cluster_data = ca.select_cluster(cluster_id)  # esto devuelve solo los objetos de ese cluster
print(f"Cluster {cluster_id} selected, contains {len(cluster_data)} sources.")

In [ ]:
ca.plot_probability_vs_gmag(figsize=(8,8)) # a esto no se la ha hecho nada, es el cluster tal cual como slió del preprocessing.

In [ ]:
pmin = 0.5
pre = (ca.data['probability_hdbscan'] >= pmin)

In [ ]:
ca.sigma_clip_parallax(
    sigma=2.0,
    use_biweight=True,
    preselector_mask=pre,
    print_results=True,
    in_place=True
)

In [ ]:
ca.plot_probability_vs_gmag(figsize=(8,8))

In [ ]:
ca.pms_characterization(
    cluster=cluster_id,   # usa el mismo cluster_id que seleccionaste (ej: 32)
    run_cli=False
)

In [ ]:
ca.plot_pms(
    cluster=cluster_id,
    pms_threshold=0.6,    # ajusta si deseas un corte distinto
    figsize=(7,7),
    layout="tight",
)

In [ ]:
cluster_data = ca.select_cluster(cluster_id)

In [ ]:
from astropy.table import QTable
cluster_data.write("../COSMIC/COSMIC/data.ecsv", format="ascii.ecsv", overwrite=True)

In [ ]:
import asteca

In [ ]:
isochs = asteca.Isochrones(isochs_path="./MIST/",
                           model='MIST',
                           magnitude="Gaia_G_EDR3",
                           magnitude_effl=6390.7,
                           color=("Gaia_BP_EDR3","Gaia_RP_EDR3"),
                           color_effl = (5182.6,7825.1),
                           #color2 = ("Gaia_RP_EDR3","2MASS_J"),
                           #color2_effl = (7825.1, 12375.60),
                          )

In [ ]:
df = cluster_data.to_pandas()

In [ ]:
my_cluster = asteca.Cluster(
    ra=df["ra"],
    dec=df["dec"],
    pmra=df["pmra"],
    pmde=df["pmdec"],
    plx=df["parallax"],
    e_pmra=df["pmra_error"],
    e_pmde=df["pmdec_error"],
    e_plx=df["parallax_error"],
    magnitude=df['Gmag'],
    e_mag=df["e_Gmag"],
    color=df["G_BPmag"] - df["G_RPmag"],
    e_color=df['e_BP_RP'],
    #color2=df["G_RPmag"] - df["j_m"],
    #e_color2 = df['e_RP_J']
)

In [ ]:
synthcl = asteca.Synthetic(isochs)

synthcl.calibrate(my_cluster)

In [ ]:
import pytensor
import pytensor.tensor as pt
from fast_histogram import histogram2d
import pymc as pm
import numpy as np

In [ ]:
# ------------- 1) BINES OBSERVADOS (rango y máscara) -----------------
# Usamos el mismo esquema que ASteCA (Tremmel); puedes cambiar 'bin_method'
from asteca.modules.likelihood_priv import lkl_data

obs_mag = my_cluster.mag                  # = cluster_data['Gmag'].value
obs_col = my_cluster.color               # = cluster_data['BP_RP'].value
ranges, Nbins, cl_z_idx, cl_histo_f_z = lkl_data(
    bin_method="knuth",                   # o "blocks"/"scott"/"freedman"/"fixed"
    mag_v=obs_mag,
    colors_v=[obs_col]
)
# Guardamos cosas útiles
mag_min = min(obs_mag.min(), syn[0].min())
mag_max = max(obs_mag.max(), syn[0].max())
col_min = min(obs_col.min(), syn[1].min())
col_max = max(obs_col.max(), syn[1].max())
Nb_mag, Nb_col   = Nbins[0], Nbins[1]
binw_mag = (mag_max - mag_min)/Nb_mag
binw_col = (col_max - col_min)/Nb_col
mask_idx = np.where(cl_z_idx)[0].astype("int32")
cl_histo_f_z = cl_histo_f_z.astype("float64")

In [ ]:
# ------------- 2) COEFICIENTES DE EXTINCIÓN (lineales) ----------------
# Opción A (preferida): usa lo que ya calculó ASteCA
# Para CCMO, synthcl.ext_coefs trae coeficientes A_lambda/Av por filtro.
# Ajusta a tu estructura exacta si difiere:
try:
    # Ejemplo típico (ajústalo según devuelva tu versión):
    # ext_coefs = (k_G, (k_BP, k_RP), (k_RP, k_J))  <-- SOLO EJEMPLO
    kG = float(synthcl.ext_coefs[0])  # coef. para magnitud G
    # primer color: BP-RP → k_BP - k_RP
    kBP, kRP = synthcl.ext_coefs[1]   # ajusta si tu estructura difiere
    k_col1 = float(kBP - kRP)
except Exception:
    # Opción B (fallback): estimarlo por diferencias finitas (rápido y práctico)
    def _est_k(delta=0.05, met_ref=None, loga_ref=None):
        if met_ref is None:  met_ref  = float(isochs.met_age_dict['met'][len(isochs.met_age_dict['met'])//2])
        if loga_ref is None: loga_ref = float(isochs.met_age_dict['loga'][len(isochs.met_age_dict['loga'])//2])
        base = synthcl.generate({"met":met_ref, "loga":loga_ref, "dm":0.0, "Av":0.0})
        pert = synthcl.generate({"met":met_ref, "loga":loga_ref, "dm":0.0, "Av":delta})
        kG_   = np.median(pert[0] - base[0]) / delta
        kcol_ = np.median(pert[1] - base[1]) / delta
        return kG_, kcol_
    kG, k_col1 = _est_k(delta=0.05)

In [ ]:
# ------------- 3) REJILLA (met, loga) Y PRE-CÁLCULO DE HESS -----------
met_min = float(isochs.met_age_dict['met'][0])
met_max = float(isochs.met_age_dict['met'][-1])
loga_min = float(isochs.met_age_dict['loga'][0])
loga_max = float(isochs.met_age_dict['loga'][-1])

# Tamaño de rejilla (ajusta según recursos)
M_MET  = 200  # puntos en metalicidad
M_LOGA = 200  # puntos en log edad

met_grid  = np.linspace(met_min,  met_max,  M_MET,  dtype=float)
loga_grid = np.linspace(loga_min, loga_max, M_LOGA, dtype=float)

# H_grid: (M_MET, M_LOGA, Nb_mag, Nb_col), histograma sintético para Av=0, DM=0
H_grid = np.zeros((M_MET, M_LOGA, Nb_mag, Nb_col), dtype=np.float64)

# Param fijos
fixed = {"alpha":0.09, "beta":0.94, "Rv":3.1, "DR":0.0}

ker = np.array([[1,2,1],
                [2,4,2],
                [1,2,1]], dtype=float)
ker /= ker.sum()

def smooth2d(H):
    # convolución 3x3 con padding cero (rápida, suficiente)
    Hpad = np.pad(H, ((1,1),(1,1)), mode="constant")
    out = np.zeros_like(H, dtype=float)
    for i in range(H.shape[0]):
        for j in range(H.shape[1]):
            block = Hpad[i:i+3, j:j+3]
            out[i,j] = np.sum(block * ker)
    return out

for i, met in enumerate(met_grid):
    for j, loga in enumerate(loga_grid):
        pars = {"met": float(met), "loga": float(loga), "dm": 0.0, "Av": 0.0} # fixed
        syn = synthcl.generate(pars)  # array: [mag, color1, (color2), mass, mass_b]
        if syn.size == 0:
            # Si queda vacío, deja todo ceros (penalizará en la PLR)
            continue
        mag_s = syn[0]
        col_s = syn[1]  # usamos el primer color (BP-RP)
        H = histogram2d(
            mag_s, col_s,
            bins=[Nb_mag, Nb_col],
            range=[[mag_min, mag_max], [col_min, col_max]],
        )
        H = smooth2d(H)
        H_grid[i, j] = H

In [ ]:
# ------------- 4) OPS PyTENSOR: interp. bilineal + corrimiento 2D -----
# 4.1 Interpolación bilineal en (met, loga) sobre H_grid (rejilla UNIFORME)
H_grid_t  = pt.as_tensor_variable(H_grid)        # (M_MET, M_LOGA, Bm, Bc)
met0, dmet   = float(met_grid[0]),  float(met_grid[1]-met_grid[0])
loga0, dloga = float(loga_grid[0]), float(loga_grid[1]-loga_grid[0])


def interp_H_met_loga(met_v, loga_v):
    mpos = (met_v  - met0)/dmet
    apos = (loga_v - loga0)/dloga

    i0 = pt.clip(pt.floor(mpos), 0, M_MET-2).astype("int64")
    j0 = pt.clip(pt.floor(apos), 0, M_LOGA-2).astype("int64")
    i1 = i0 + 1
    j1 = j0 + 1

    wm = mpos - i0
    wa = apos - j0

    H00 = H_grid_t[i0, j0]
    H10 = H_grid_t[i1, j0]
    H01 = H_grid_t[i0, j1]
    H11 = H_grid_t[i1, j1]
    return ((1-wm)*(1-wa))*H00 + (wm*(1-wa))*H10 + ((1-wm)*wa)*H01 + (wm*wa)*H11  # (Bm,Bc)

# ---- NUEVO: corrimiento bilineal “pull” JAX-safe, sin set_subtensor ----
def _meshgrid_float(m, n):
    I = pt.arange(m, dtype="float64").dimshuffle(0, "x")
    J = pt.arange(n, dtype="float64").dimshuffle("x", 0)
    return I, J

def _gather2d(A, Ui, Vj):
    Ui = pt.clip(Ui, 0, A.shape[0]-1).astype("int32")
    Vj = pt.clip(Vj, 0, A.shape[1]-1).astype("int32")
    return A[Ui, Vj]

def shift_histogram(H, dmag, dcol):
    # desplazamientos en unidades de bin
    su = dmag / binw_mag  # eje magnitud (filas)
    sv = dcol / binw_col  # eje color (columnas)

    m = H.shape[0]
    n = H.shape[1]
    I, J = _meshgrid_float(m, n)

    # coordenadas fuente (de dónde interpolar)
    U = I - su
    V = J - sv

    u0 = pt.floor(U); v0 = pt.floor(V)
    fu = (U - u0).astype("float64")
    fv = (V - v0).astype("float64")
    u0i = u0.astype("int32"); v0j = v0.astype("int32")
    u1i = (u0 + 1).astype("int32"); v1j = (v0 + 1).astype("int32")

    # máscaras de validez (si el punto fuente cae fuera → 0)
    valid = (
        (U >= 0) & (U <= (m - 1)) &
        (V >= 0) & (V <= (n - 1))
    ).astype("float64")

    H00 = _gather2d(H, u0i, v0j)
    H10 = _gather2d(H, u1i, v0j)
    H01 = _gather2d(H, u0i, v1j)
    H11 = _gather2d(H, u1i, v1j)

    w00 = (1 - fu) * (1 - fv)
    w10 = fu * (1 - fv)
    w01 = (1 - fu) * fv
    w11 = fu * fv

    out = w00*H00 + w10*H10 + w01*H01 + w11*H11
    return out * valid  # cero si todo cae fuera

# 4.3 Log-PLR (Tremmel) usando sólo bins observados != 0
cl_histo_f_z_t = pt.as_tensor_variable(cl_histo_f_z)
mask_idx_t = pt.as_tensor_variable(mask_idx.astype("int64"))        # índices de bins != 0
obs_vec = cl_histo_full.ravel()                    # vector observado
cl_obs_t   = pt.as_tensor_variable(cl_obs_vec)
obs_t = pt.as_tensor_variable(obs_vec)
def my_logp_fn(value, met, loga, dm, Av):
    """
    value: vector observado (cl_histo_f_z) con los conteos en bins != 0
    Devuelve la log-PLR de Tremmel: sum[ logΓ(n_i+m_i+1/2) - logΓ(m_i+1/2) ]
    """
    # 1) hist sintético base (met, loga)
    H0   = interp_H_met_loga(met, loga)           # (Nb_mag, Nb_col)
    # 2) corrimientos por distancia y extinción (lineales)
    dmag = dm + kG * Av
    dcol = k_col1 * Av
    Hsyn = shift_histogram(H0, dmag, dcol)        # (Nb_mag, Nb_col)

    # 3) aplana y toma solo los bins observados
    syn_f_z = Hsyn.reshape((-1,))[mask_idx_t]     # vector (Nb_keep,)

    # 4) Tremmel log-PLR
    v = pt.as_tensor_variable(value)              # asegurar tensor
    return pt.sum(pt.gammaln(v + syn_f_z + 0.5) - pt.gammaln(syn_f_z + 0.5))

def log_plr_from_hist(H_syn):
    syn_flat = H_syn.reshape((-1,))
    syn_f_z  = syn_flat[mask_idx_t]
    return pt.sum(pt.gammaln(cl_histo_f_z_t + syn_f_z + 0.5) - pt.gammaln(syn_f_z + 0.5))

In [ ]:
with pm.Model() as model:
    # Priors
    met  = pm.Uniform("met",  lower=met_min,  upper=met_max)
    loga = pm.Uniform("loga", lower=loga_min, upper=loga_max)
    dm   = pm.TruncatedNormal("dm", mu=10.2, sigma=0.3, lower=9.5, upper=10.7)
    Av   = pm.Uniform("Av", lower=0.0, upper=3.0)

    # H interpolado y desplazado
    H0   = interp_H_met_loga(met, loga)            # (Nb_mag, Nb_col)
    dmag = dm + kG * Av
    dcol = k_col1 * Av
    Hsyn = shift_histogram(H0, dmag, dcol)         # (Nb_mag, Nb_col)

    lam  = Hsyn.reshape((-1,))                     # todos los bins
    eps = 1e-6
    lam = pt.clip(lam, eps, np.inf)

    # --- NUEVO: escala y fondo ---
    log_s = pm.Normal("log_s", mu=0.0, sigma=2.0)  # factor global
    bg    = pm.HalfNormal("bg", sigma=0.2)         # cuenta por bin (ajústalo)
    mu    = pt.exp(log_s) * lam + bg               # (Nb_mag*Nb_col,)

    y = pm.Poisson("y", mu=mu, observed=obs_t)

In [ ]:
with model:
    idata = pm.sample(
        draws=20000, tune=10000,
        nuts_sampler="blackjax",target_accept=0.95
    )

In [ ]:
import arviz as az

# Resumen numérico de las 4 params principales
az.summary(idata, var_names=["met","loga","dm","Av"], kind="stats", round_to=4)

In [ ]:
# Posterior univariado
az.plot_posterior(idata, var_names=["met","loga","dm","Av"], hdi_prob=0.94);

In [ ]:
# Corner plot (pares), con KDE
az.plot_pair(
    idata,
    var_names=["met","loga","dm","Av"],
    kind="kde",  # o "scatter"
    divergences="bottom",
    marginals=True
);

In [ ]:
# Trazas por cadena (útil para ver mezcla)
az.plot_trace(idata, var_names=["met","loga","dm","Av"]);

In [ ]:
import numpy as np

post = idata.posterior

met_m  = np.median(np.asarray(post["met"]).reshape(-1))
loga_m = np.median(np.asarray(post["loga"]).reshape(-1))
dm_m   = np.median(np.asarray(post["dm"]).reshape(-1))
Av_m   = np.median(np.asarray(post["Av"]).reshape(-1))

best = {"met": float(met_m),
        "loga": float(loga_m),
        "dm": float(dm_m),
        "Av": float(Av_m)}

print(best)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- adaptador de compatibilidad ---
def generate_compat(synthcl, params, plot_flag=False):
    try:
        # firma vieja (dict + plot_flag)
        return synthcl.generate(params, plot_flag=plot_flag)
    except TypeError:
        try:
            # firma más común actual (solo dict)
            return synthcl.generate(params)
        except TypeError:
            # algunas implementaciones esperan kwargs sueltos
            return synthcl.generate(**params)

# Si ya tienes 'best' y 'fixed' construidos:
pars = dict(best)  # met, loga, dm, Av
pars.update(fixed) # alpha, beta, Rv, DR, etc.

syn = generate_compat(synthcl, pars, plot_flag=False)

# ASteCA suele devolver: [mag, color1, (color2), ...]
mag_s = syn[0]
col_s = syn[1]

# Observados (ajusta a tus arrays reales)
mag_o = np.asarray(my_cluster.mag)
col_o = np.asarray(my_cluster.color)

# --- plot rápido CMD ---
plt.figure(figsize=(5,6))
plt.scatter(col_o, mag_o, s=8, alpha=0.35, label="Obs")
plt.scatter(col_s, mag_s, s=8, alpha=0.35, label="Synth @ posterior mediana")
plt.gca().invert_yaxis()
plt.xlabel("BP−RP")
plt.ylabel("G")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import arviz as az

az.summary(idata, hdi_prob=0.95)

In [ ]:
post_median = {v: np.median(idata.posterior[v].values.flatten())
               for v in ["met", "dm", "loga", "Av"]}
print(post_median)

In [ ]:
import arviz as az
import numpy as np
import matplotlib.pyplot as plt

ds = az.extract(idata, num_samples=20, var_names=["met", "dm", "loga", "Av"])  # xarray.Dataset

plt.figure()
for i in range(ds.sizes["sample"]):  # número de muestras extraídas
    pars = {k: float(ds[k].values[i]) for k in ds.data_vars}  # dict con floats
    syn = synthcl.generate(pars)  # [mag, color1, ...]
    plt.scatter(syn[1], syn[0], s=3, alpha=0.3, label="posterior draw" if i == 0 else "")

plt.gca().invert_yaxis()
plt.legend()
plt.show()

In [ ]:
print(az.summary(idata, var_names=["met","loga","dm","Av"], kind="stats"))

# --- percentiles útiles ---
p = az.extract(idata, var_names=["met","loga","dm","Av"]).to_dataframe()
p50 = p.median()  # mediana marginal (baseline)
best_now = dict(met=float(p50["met"]), loga=float(p50["loga"]),
                dm=float(p50["dm"]),  Av=float(p50["Av"]))
print("Baseline (mediana):", best_now)

In [ ]:
# 1) puntos observados (lo que venías usando)
x_obs = my_cluster.color   # BP-RP
y_obs = my_cluster.mag     # G

# 2) sintético a la mediana del posterior
pars_syn = best_now | {"alpha":0.09, "beta":0.94, "Rv":3.1, "DR":0.0}
syn = synthcl.generate(pars_syn)  # devuelve [G, BP-RP, (opcional otros)]
x_syn, y_syn = syn[1], syn[0]

plt.figure(figsize=(6,7))
plt.scatter(x_obs, y_obs, s=7, alpha=0.5, label="Obs", zorder=2)
plt.scatter(x_syn, y_syn, s=7, alpha=0.5, label="Synth @mediana", zorder=3)
plt.gca().invert_yaxis()
plt.xlabel("BP−RP"); plt.ylabel("G"); plt.legend(); plt.title("Baseline CMD")
plt.tight_layout(); plt.show()

In [ ]:
# hist observado completo (misma rejilla que el modelo)
cl_histo_full = histogram2d(
    obs_mag, obs_col,
    bins=[Nb_mag, Nb_col],
    range=[[mag_min, mag_max], [col_min, col_max]],
).astype("float64")

# hist sintético desde H_grid interpolado + shift (lo mismo que en el modelo)
H0 = interp_H_met_loga(best_now["met"], best_now["loga"])
dmag = best_now["dm"] + kG*best_now["Av"]
dcol = k_col1*best_now["Av"]
Hsyn = shift_histogram(H0, dmag, dcol).eval()  # tensor → numpy

# plots comparativos
fig,ax = plt.subplots(1,3,figsize=(12,4),sharex=True,sharey=True)
im0=ax[0].imshow(cl_histo_full.T, origin="lower", aspect="auto")
ax[0].set_title("Obs bins")
im1=ax[1].imshow(Hsyn.T, origin="lower", aspect="auto")
ax[1].set_title("Synth bins @mediana")
diff = cl_histo_full - Hsyn
im2=ax[2].imshow(diff.T, origin="lower", aspect="auto")
ax[2].set_title("Obs − Synth")
for a in ax: a.set_xlabel("mag bin"); a.set_ylabel("color bin")
plt.tight_layout(); plt.show()

In [ ]:
print(syn[0].min(), syn[0].max())   # magnitudes sintéticas
print(syn[1].min(), syn[1].max())   # colores sintéticos

In [ ]:
print(obs_mag.min(), obs_mag.max())
print(obs_col.min(), obs_col.max())

In [ ]:
# loglike promedio (del sample)
llk = idata.sample_stats["lp"].mean().item()
print(f"logp medio (baseline): {llk:.1f}")

# LOO/WAIC si usaste Poisson observado:
try:
    loo_base = az.loo(idata)
    waic_base = az.waic(idata)
    print("LOO (baseline):", loo_base)
    print("WAIC(baseline):", waic_base)
except Exception as e:
    print("LOO/WAIC no disponible para esta estructura:", e)